# Baseline CIFAR-10 và phép đo vanishing gradient

Notebook bám mục **6 — Ví dụ minh họa**, phần **Cấu hình mô hình baseline**
trong `vanishing-gradient/vanishing_gradient_outline.tex` (Bảng cấu hình CIFAR-10).
Mục tiêu là chạy một CNN sâu đơn giản, lưu baseline ổn định và đo các cơ chế
trong mục 5 của report. Baseline có BatchNorm và shortcut nên **không nhất thiết
xuất hiện vanishing gradient**; các ablation mới giúp đánh giá tác động từng yếu tố.

| Thành phần | Cấu hình report |
|---|---|
| Dữ liệu | CIFAR-10, train/validation/test = 45.000/5.000/10.000; validation phân tầng |
| Chuẩn hóa | Mean/std từng kênh tính trên 45.000 ảnh train, trước augmentation |
| Augmentation | RandomCrop(32, padding=4), RandomHorizontalFlip, chỉ trên train |
| Stem | Conv 3×3, 3→64 kênh, stride=1, padding=1 |
| Backbone | 8 block, mỗi block Conv 3×3 → BatchNorm → ReLU → cộng identity |
| Head | Global average pooling → Linear(64, 10); trả về logits |
| Khởi tạo | He normal cho Conv và Linear, bias=0; BN scale=1, bias=0 |
| Huấn luyện | CrossEntropy; SGD(lr=0.1, momentum=0.9, weight_decay=5e-4) |
| Lịch | 100 epoch, batch=128, cosine LR đến 0, không warm restart |
| Lặp | Seed 0, 1, 2; báo cáo mean ± sample std (ddof=1) |

**Cách chạy:** mở notebook bằng Jupyter/VS Code hoặc Google Colab, cài thư viện ở
cell tiếp theo nếu cần, chọn GPU và Run All. `PROFILE="report"` là mặc định đúng
report. Đổi sang `"quick"` để kiểm tra pipeline trên tập con trong 2 epoch, 1 seed;
kết quả quick không dùng làm kết quả report. Chạy report đầy đủ tốn nhiều tài nguyên
vì mọi block giữ nguyên kích thước 32×32. `RUN_ABLATIONS=False` mặc định chỉ chạy baseline.

File dữ liệu/log/checkpoint/ảnh được ghi vào `output/vanishing_gradient/` ở root
project (hoặc thư mục hiện tại nếu chạy trên Colab). Test được dùng một lần ở cell
đánh giá cuối, sau khi chọn checkpoint bằng validation.


In [1]:
# Bỏ dấu # và chạy nếu kernel chưa có thư viện. Trên Colab thường chỉ cần các gói phụ.
%pip install torch torchvision numpy pandas matplotlib ipykernel nbformat nbclient


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 74.2 MB/s eta 0:00:00a 0:00:01


In [2]:
import copy
import hashlib
import json
import math
import os
import platform
import random
import time
from dataclasses import asdict, dataclass, replace
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torchvision
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
from IPython.display import display

plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True, "grid.alpha": 0.25})
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
OUTPUT_ROOT = ROOT / "output" / "vanishing_gradient"
DATA_ROOT = OUTPUT_ROOT / "data"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else
                      "mps" if torch.backends.mps.is_available() else "cpu")
print(f"PyTorch {torch.__version__}; torchvision {torchvision.__version__}; device={DEVICE}")
print("Output:", OUTPUT_ROOT)


PyTorch 2.11.0+cu130; torchvision 0.26.0+cu130; device=cuda
Output: /content/output/vanishing_gradient


## 1. Cấu hình và tái lập thí nghiệm

Split có seed riêng cố định cho mọi lần chạy. Seed mô hình/dataloader là 0,1,2.
DataLoader `num_workers=0` để chạy được trên notebook Windows/macOS/Colab và giữ
augmentation có thể tái lập. RNG của huấn luyện được đặt lại mỗi epoch, tách khỏi
RNG khởi tạo mô hình, nên các ablation dùng cùng thứ tự ảnh và augmentation ở cùng seed.
Không dùng AMP, clipping hoặc accumulation để tránh thay đổi thang đo gradient.
Độ chính xác tuyệt đối giữa các thiết bị/phiên bản PyTorch vẫn có thể khác.


In [3]:
@dataclass(frozen=True)
class ModelConfig:
    depth: int = 8
    width: int = 64
    bottleneck: int | None = None  # Một block giữa mạng thu hẹp output channels.
    activation: str = "relu"
    initialization: str = "he"
    normalization: str = "batchnorm"
    residual: bool = True

@dataclass(frozen=True)
class TrainConfig:
    epochs: int = 100
    batch_size: int = 128
    lr: float = 0.1
    momentum: float = 0.9
    weight_decay: float = 5e-4
    seeds: tuple = (0, 1, 2)
    split_seed: int = 2026
    probe_batch_size: int = 32
    probe_every: int = 1
    num_workers: int = 0

BASELINE = ModelConfig()
REPORT = TrainConfig()
PROFILE = "report"  # Đổi thành "quick" để thử pipeline.
assert PROFILE in {"report", "quick"}
TRAIN = REPORT if PROFILE == "report" else replace(REPORT, epochs=2, seeds=(0,))
QUICK_TRAIN_PER_CLASS = 100
QUICK_VAL_PER_CLASS = 50
RUN_ABLATIONS = False
ABLATION_NAMES = ["no_shortcut"]  # Chọn thêm tên ở cell ablation nếu cần.
EVALUATE_TEST = PROFILE == "report"
SELECTED_EXPERIMENT = "baseline"  # Quyết định bằng validation, trước khi nhìn test.
RUN_ROOT = OUTPUT_ROOT / (PROFILE + "_" + time.strftime("%Y%m%d_%H%M%S") + f"_{time.time_ns() % 1000000:06d}")
RUN_ROOT.mkdir(parents=True, exist_ok=False)

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed % (2**32))
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
if hasattr(torch.backends.cuda.matmul, "allow_tf32"):
    torch.backends.cuda.matmul.allow_tf32 = False
torch.use_deterministic_algorithms(True, warn_only=True)
display(pd.DataFrame([asdict(BASELINE)]))
print("Profile:", PROFILE, "| epochs:", TRAIN.epochs, "| seeds:", TRAIN.seeds)
print("Run:", RUN_ROOT)


,depth,width,bottleneck,activation,initialization,normalization,residual
0,8,64,None,relu,he,batchnorm,True


Profile: report | epochs: 100 | seeds: (0, 1, 2)
Run: /content/output/vanishing_gradient/report_20261006_101501_398825


## 2. CIFAR-10: split và chuẩn hóa không rò rỉ dữ liệu

Tách chính xác 500 ảnh mỗi lớp làm validation. Mean/std theo pixel tính bằng
float64, chỉ trên 45.000 ảnh train; không dùng validation/test để tính thống kê.
Quick lấy tập con phân tầng từ split này nhưng vẫn dùng bộ thống kê train của
report. Batch probe lấy ảnh train cố định, không augmentation, cân bằng lớp;
nhãn và ảnh probe giống nhau giữa các seed/cấu hình.


In [ ]:
raw_train = datasets.CIFAR10(root=str(DATA_ROOT), train=True, download=True)
targets = np.asarray(raw_train.targets)
rng = np.random.default_rng(TRAIN.split_seed)
train_indices, val_indices = [], []
for label in range(10):
    indices = rng.permutation(np.flatnonzero(targets == label))
    val_indices.extend(indices[:500])
    train_indices.extend(indices[500:])
train_indices = np.asarray(train_indices, dtype=np.int64)
val_indices = np.asarray(val_indices, dtype=np.int64)
assert len(train_indices) == 45000 and len(val_indices) == 5000
assert not np.intersect1d(train_indices, val_indices).size
assert np.array_equal(np.bincount(targets[val_indices]), np.full(10, 500))
np.savez(RUN_ROOT / "split.npz", train=train_indices, validation=val_indices)

def channel_statistics(images, indices, chunk_size=1000):
    total, total_sq = np.zeros(3), np.zeros(3)
    pixel_count = 0
    for start in range(0, len(indices), chunk_size):
        x = images[indices[start:start + chunk_size]].astype(np.float64) / 255.0
        total += x.sum(axis=(0, 1, 2))
        total_sq += (x * x).sum(axis=(0, 1, 2))
        pixel_count += np.prod(x.shape[:3])
    mean = total / pixel_count
    std = np.sqrt(np.maximum(total_sq / pixel_count - mean**2, 0))
    assert np.all(std > 0)
    return mean.tolist(), std.tolist()

mean, std = channel_statistics(raw_train.data, train_indices)
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip(),
    transforms.ToTensor(), transforms.Normalize(mean, std),
])
eval_transform = transforms.Compose([
    transforms.ToTensor(), transforms.Normalize(mean, std),
])
train_dataset = datasets.CIFAR10(str(DATA_ROOT), train=True, transform=train_transform)
eval_dataset = datasets.CIFAR10(str(DATA_ROOT), train=True, transform=eval_transform)

def balanced_subset(indices, per_class):
    return np.concatenate([indices[targets[indices] == c][:per_class] for c in range(10)])

used_train = train_indices if PROFILE == "report" else balanced_subset(train_indices, QUICK_TRAIN_PER_CLASS)
used_val = val_indices if PROFILE == "report" else balanced_subset(val_indices, QUICK_VAL_PER_CLASS)
probe_indices = balanced_subset(train_indices, math.ceil(TRAIN.probe_batch_size / 10))
# Xen kẽ lớp để batch probe không dồn các lớp vào đầu danh sách.
probe_indices = probe_indices.reshape(10, -1).T.flatten()[:TRAIN.probe_batch_size]
probe_x = torch.stack([eval_dataset[int(i)][0] for i in probe_indices])
probe_y = torch.tensor(targets[probe_indices], dtype=torch.long)
np.savez(RUN_ROOT / "used_indices.npz", train=used_train, validation=used_val, probe=probe_indices)

def make_loaders(seed):
    generator = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(Subset(train_dataset, used_train.tolist()),
        batch_size=TRAIN.batch_size, shuffle=True, generator=generator,
        num_workers=TRAIN.num_workers, pin_memory=DEVICE.type == "cuda")
    val_loader = DataLoader(Subset(eval_dataset, used_val.tolist()),
        batch_size=TRAIN.batch_size, shuffle=False, num_workers=TRAIN.num_workers)
    return train_loader, val_loader

print("Mean:", mean, "| std:", std)
print(f"Dữ liệu dùng: train={len(used_train)}, validation={len(used_val)}, probe={len(probe_y)}")
display(pd.DataFrame({"class": raw_train.classes,
    "train": np.bincount(targets[used_train], minlength=10),
    "validation": np.bincount(targets[used_val], minlength=10)}))


  4%|▍         | 6.49M/170M [01:23<36:40, 74.5kB/s]

## 3. CNN mô-đun tối giản

Stem chỉ có Conv, không thêm activation/BN ngoài report. Mỗi block đúng một
Conv 3×3 → normalization → activation → cộng shortcut, **không ReLU sau phép cộng**.
Giữ stride=1 và padding=1 ở mọi block. Conv có bias để cấu trúc tham số không đổi
khi thay activation, khởi tạo hoặc bật/tắt shortcut cùng kích thước.

LayerNorm tùy chọn chuẩn hóa toàn bộ `(C,H,W)` của từng ảnh, dùng affine theo kênh
để số tham số affine bằng BatchNorm (không phải LayerNorm chỉ theo C). `none` bỏ
chuẩn hóa và affine nên ít hơn 2C tham số/block; ghi rõ chênh lệch này trong bảng
chi phí. Đây là chi tiết report chưa đặc tả. Bottleneck đặt output block giữa mạng
bằng 16 kênh, block sau phục hồi độ rộng; dùng projection 1×1 ở hai chỗ đổi chiều.


In [ ]:
class ChannelAffineLayerNorm(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(1, channels, 1, 1))
        self.bias = nn.Parameter(torch.zeros(1, channels, 1, 1))

    def forward(self, x):
        return F.layer_norm(x, x.shape[1:]) * self.weight + self.bias

def make_activation(name):
    return {"relu": nn.ReLU, "sigmoid": nn.Sigmoid, "tanh": nn.Tanh}[name]()

def make_norm(name, channels):
    if name == "batchnorm":
        return nn.BatchNorm2d(channels)
    if name == "layernorm":
        return ChannelAffineLayerNorm(channels)
    if name == "none":
        return nn.Identity()
    raise ValueError(name)

class Block(nn.Module):
    def __init__(self, in_channels, out_channels, cfg):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=True)
        self.norm = make_norm(cfg.normalization, out_channels)
        self.activation = make_activation(cfg.activation)
        self.residual = cfg.residual
        self.shortcut = (nn.Conv2d(in_channels, out_channels, 1, bias=True)
                         if cfg.residual and in_channels != out_channels else nn.Identity())

    def forward(self, x, capture=False):
        pre = self.conv(x)
        z = self.norm(pre)
        a = self.activation(z)
        h = a + self.shortcut(x) if self.residual else a
        if capture:
            # Non-leaf tensors cần retain_grad để đọc gradient sau backward.
            for tensor in (pre, z, a, h):
                tensor.retain_grad()
            return h, {"pre": pre, "z": z, "a": a, "h": h}
        return h, None

class DeepCNN(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert cfg.depth > 0 and cfg.width > 0
        self.cfg = cfg
        self.stem = nn.Conv2d(3, cfg.width, 3, padding=1, bias=True)
        channels = [cfg.width] * cfg.depth
        if cfg.bottleneck is not None:
            channels[cfg.depth // 2] = cfg.bottleneck
        blocks, previous = [], cfg.width
        for current in channels:
            blocks.append(Block(previous, current, cfg))
            previous = current
        self.blocks = nn.ModuleList(blocks)
        self.head = nn.Linear(previous, 10)
        self.reset_weights()

    def reset_weights(self):
        for module in self.modules():
            if isinstance(module, (nn.Conv2d, nn.Linear)):
                if self.cfg.initialization == "he":
                    nn.init.kaiming_normal_(module.weight, mode="fan_in", nonlinearity="relu")
                elif self.cfg.initialization == "xavier":
                    nn.init.xavier_normal_(module.weight, gain=1.0)
                elif self.cfg.initialization == "small_gaussian":
                    nn.init.normal_(module.weight, mean=0.0, std=0.01)
                else:
                    raise ValueError(self.cfg.initialization)
                nn.init.zeros_(module.bias)
            elif isinstance(module, (nn.BatchNorm2d, ChannelAffineLayerNorm)):
                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)

    def forward(self, x, capture=False):
        h = self.stem(x)
        trace = []
        if capture:
            h.retain_grad()
            trace.append({"h": h})
        for block in self.blocks:
            h, entry = block(h, capture)
            if capture:
                trace.append(entry)
        logits = self.head(h.mean(dim=(2, 3)))
        return (logits, trace) if capture else logits

seed_all(0)
preview_model = DeepCNN(BASELINE)
with torch.no_grad():
    assert preview_model(torch.zeros(2, 3, 32, 32)).shape == (2, 10)
print(preview_model)
print("Parameters:", sum(p.numel() for p in preview_model.parameters()))
del preview_model


## 4. Phép đo nguyên nhân và dấu hiệu

| Cơ chế trong report | Phép đo trong notebook | Cách đọc |
|---|---|---|
| Chuỗi biến đổi làm co gradient | L2 và RMS của `∂L/∂h` tại stem và sau từng block; tỉ lệ stem/block cuối | Gradient trạng thái giảm nhiều bậc về đầu mạng là dấu hiệu trực tiếp |
| Đạo hàm activation nhỏ/bão hòa | Mean `φ'(z)`, mean `φ'(z)²`, tỉ lệ `<0.01`, histogram z/activation | Sigmoid/tanh: đạo hàm nhỏ nhiều; ReLU: phần bị chặn tại z≤0 |
| Khởi tạo quá nhỏ/lớn | RMS weight, RMS preactivation trước norm, RMS z sau norm, RMS activation | Phân biệt tín hiệu nhỏ với bão hòa; BN có thể che ảnh hưởng scale weight |
| Mất khác biệt giữa đầu vào | Phương sai activation theo batch và RMS khoảng cách cặp ảnh khác nhãn | Giảm phương sai/khoảng cách chỉ là proxy, không đo Shannon information |
| Lớp đầu ít được cập nhật | `||W_t−W_0||₂/||W_0||₂`, gradient weight L2/RMS | Đọc cùng gradient trạng thái; gradient weight có thể nhỏ do activation hoặc triệt tiêu trong batch |
| Độ sâu/rộng, norm, shortcut | Ablation một yếu tố; cùng split, seed, dữ liệu và lịch LR | Baseline đơn lẻ chỉ cho tương quan; so sánh có kiểm soát mới hỗ trợ quy tác động |

Probe riêng ở epoch 0 và sau mỗi epoch: **model.train()**, ảnh cố định không augment,
CrossEntropy `reduction='mean'`, `backward()` và **không optimizer.step()**. Khôi phục
mọi buffer BN, mode và RNG sau probe để phép đo không thay đổi huấn luyện. Norm/RMS
tính trên CPU float64 từ gradient float32 đã sinh; không khôi phục được gradient
đã underflow về 0. Theo dõi cả tỉ lệ gradient bằng 0. Batch probe nhỏ hơn batch train
nên thống kê là phép chẩn đoán chuẩn hóa, không phải trung bình gradient tất cả batch train.

`∂L/∂h` lấy **sau phép cộng residual**, còn `∂L/∂z` và derivative đo ở **nhánh residual**.
Đạo hàm nhánh nhỏ không đồng nghĩa gradient cả block nhỏ khi vẫn có identity path.
RMS giúp so sánh độ rộng; L2 còn phụ thuộc số phần tử. Nếu gradient mẫu số bằng 0,
tỉ lệ ghi NaN, không thêm epsilon để tạo một tỉ lệ có vẻ hợp lệ. Với ReLU, thống kê
neuron bất hoạt trên probe không đủ kết luận neuron chết trên toàn tập dữ liệu.


In [ ]:
def cpu64(tensor):
    return tensor.detach().to(device="cpu", dtype=torch.float64)

def rms(tensor):
    t = cpu64(tensor)
    return t.square().mean().sqrt().item()

def l2(tensor):
    return torch.linalg.vector_norm(cpu64(tensor)).item()

def ratio(numerator, denominator):
    return numerator / denominator if denominator > 0 else float("nan")

def activation_derivative(z, activation):
    z = cpu64(z)
    if activation == "relu":
        return (z > 0).double()
    if activation == "sigmoid":
        a = torch.sigmoid(z)
        return a * (1 - a)
    return 1 - torch.tanh(z).square()

def sample_values(tensor, max_items=4096):
    flat = tensor.detach().float().cpu().flatten()
    step = max(1, math.ceil(flat.numel() / max_items))
    return flat[::step].numpy().copy()

def state_diagnostics(model, initial_weights, epoch, save_samples=False):
    modes = {module: module.training for module in model.modules()}
    buffers = {name: value.detach().clone() for name, value in model.named_buffers()}
    py_rng, np_rng = random.getstate(), np.random.get_state()
    cpu_rng = torch.get_rng_state()
    cuda_rng = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
    mps_rng = torch.mps.get_rng_state() if DEVICE.type == "mps" else None
    rows, weight_rows, samples = [], [], {}
    try:
        model.train()
        model.zero_grad(set_to_none=True)
        logits, trace = model(probe_x.to(DEVICE), capture=True)
        loss = F.cross_entropy(logits, probe_y.to(DEVICE))
        if not torch.isfinite(loss):
            raise FloatingPointError("Probe loss không hữu hạn; kiểm tra exploding gradient.")
        loss.backward()
        for i, entry in enumerate(trace):
            h, grad = entry["h"], entry["h"].grad
            if not torch.isfinite(grad).all():
                raise FloatingPointError(f"Gradient không hữu hạn ở layer {i}")
            values = cpu64(h)
            # Các cặp kề nhau trong probe có nhãn khác nhau.
            row = {"epoch": epoch, "layer": i, "name": "stem" if i == 0 else f"block_{i:02d}",
                "h_grad_l2": l2(grad), "h_grad_rms": rms(grad),
                "h_grad_zero_fraction": (grad == 0).float().mean().item(),
                "h_rms": rms(h), "h_batch_var": values.var(dim=0, unbiased=False).mean().item(),
                "h_pair_distance_rms": (values[::2][:len(values)//2] - values[1::2]).square().mean().sqrt().item(),
                "probe_loss": loss.item()}
            if i:
                derivative = activation_derivative(entry["z"], model.cfg.activation)
                a = cpu64(entry["a"])
                row.update(pre_norm_rms=rms(entry["pre"]), z_rms=rms(entry["z"]),
                    a_rms=rms(entry["a"]), z_grad_rms=rms(entry["z"].grad),
                    derivative_mean=derivative.mean().item(),
                    derivative_sq_mean=derivative.square().mean().item(),
                    small_derivative_fraction=(derivative < 0.01).double().mean().item(),
                    relu_inactive_unit_fraction=(a.amax(dim=(0, 2, 3)) == 0).double().mean().item()
                        if model.cfg.activation == "relu" else float("nan"))
            rows.append(row)
            if save_samples:
                for key, tensor in entry.items():
                    samples[f"layer_{i:02d}_{key}"] = sample_values(tensor)
                samples[f"layer_{i:02d}_h_grad"] = sample_values(grad)
        for name, parameter in model.named_parameters():
            if parameter.ndim < 2 or not name.endswith("weight"):
                continue  # Chỉ Conv/Linear weights (LN affine cũng loại bằng tên).
            if name.endswith("norm.weight"):
                continue
            base = initial_weights[name]
            grad = parameter.grad
            weight_rows.append({"epoch": epoch, "name": name,
                "w_l2": l2(parameter), "w_rms": rms(parameter),
                "w_grad_l2": l2(grad), "w_grad_rms": rms(grad),
                "w_grad_zero_fraction": (grad == 0).float().mean().item(),
                "relative_drift": ratio(l2(cpu64(parameter) - base.double()), l2(base))})
        summary = {"epoch": epoch, "probe_loss": loss.item(),
            "state_first_last_l2_ratio": ratio(rows[0]["h_grad_l2"], rows[-1]["h_grad_l2"]),
            "state_first_last_rms_ratio": ratio(rows[0]["h_grad_rms"], rows[-1]["h_grad_rms"])}
        lookup = {row["name"]: row for row in weight_rows}
        first, last = lookup["stem.weight"], lookup[f"blocks.{model.cfg.depth - 1}.conv.weight"]
        summary["weight_first_last_l2_ratio"] = ratio(first["w_grad_l2"], last["w_grad_l2"])
        summary["weight_first_last_rms_ratio"] = ratio(first["w_grad_rms"], last["w_grad_rms"])
        summary["weight_stem_head_l2_ratio"] = ratio(first["w_grad_l2"], lookup["head.weight"]["w_grad_l2"])
        return rows, weight_rows, summary, samples
    finally:
        with torch.no_grad():
            for name, value in model.named_buffers():
                value.copy_(buffers[name])
        for module, training in modes.items():
            module.training = training
        model.zero_grad(set_to_none=True)
        random.setstate(py_rng)
        np.random.set_state(np_rng)
        torch.set_rng_state(cpu_rng)
        if cuda_rng is not None:
            torch.cuda.set_rng_state_all(cuda_rng)
        if mps_rng is not None:
            torch.mps.set_rng_state(mps_rng)

def compute_cost(model):
    # MACs mỗi ảnh; Conv/Linear, không tính BN/activation/cộng/pooling/backward.
    macs, handles = [], []
    def hook(module, inputs, output):
        if isinstance(module, nn.Conv2d):
            macs.append(output[0].numel() * module.weight[0].numel())
        else:
            macs.append(module.in_features * module.out_features)
    training = model.training
    try:
        model.eval()
        for module in model.modules():
            if isinstance(module, (nn.Conv2d, nn.Linear)):
                handles.append(module.register_forward_hook(hook))
        with torch.no_grad():
            model(torch.zeros(1, 3, 32, 32, device=next(model.parameters()).device))
    finally:
        for handle in handles:
            handle.remove()
        model.train(training)
    return {"parameters": sum(p.numel() for p in model.parameters()),
            "conv_linear_macs_per_image": sum(macs)}


## 5. Huấn luyện và lưu kết quả theo từng seed

Train loss/accuracy trên ảnh có augmentation; validation không augmentation.
Probe chạy ở epoch 0 và sau huấn luyện mỗi epoch. Lưu checkpoint **validation
accuracy cao nhất**, không early stopping, vẫn đủ 100 epoch. Các CSV cập nhật sau
mỗi epoch để giữ số đo khi notebook bị ngắt; checkpoint chỉ chứa model/config,
không có optimizer state để resume. Drift đo từ khởi tạo, gồm ảnh hưởng momentum
và weight decay; nó không phải riêng độ lớn bước cập nhật từ data gradient.

Optimizer sử dụng gradient train bình thường; không dùng probe gradient để cập
nhật. Probe được zero_grad trước/sau và không cộng weight decay vào gradient ghi log.


In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss, correct, count = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        total_loss += F.cross_entropy(logits, y, reduction="sum").item()
        correct += (logits.argmax(1) == y).sum().item()
        count += len(y)
    return {"loss": total_loss / count, "accuracy": correct / count}

def train_epoch(model, loader, optimizer):
    model.train()
    total_loss, correct, count = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = F.cross_entropy(logits, y)
        if not torch.isfinite(loss):
            raise FloatingPointError("Train loss không hữu hạn; run bị dừng, không kết luận vanishing.")
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y)
        correct += (logits.detach().argmax(1) == y).sum().item()
        count += len(y)
    return {"loss": total_loss / count, "accuracy": correct / count}

def run_experiment(name, cfg, seed):
    seed_all(seed)
    directory = RUN_ROOT / name / f"seed_{seed}"
    directory.mkdir(parents=True, exist_ok=False)
    model = DeepCNN(cfg).to(DEVICE)
    initial = {n: p.detach().cpu().clone() for n, p in model.named_parameters() if n.endswith("weight")}
    cost = compute_cost(model)
    train_loader, val_loader = make_loaders(seed)
    optimizer = torch.optim.SGD(model.parameters(), lr=TRAIN.lr,
        momentum=TRAIN.momentum, weight_decay=TRAIN.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TRAIN.epochs, eta_min=0)
    history, states, weights, ratios = [], [], [], []
    metadata = {"name": name, "seed": seed, "profile": PROFILE, "model": asdict(cfg),
        "train": asdict(TRAIN), "normalization_mean": mean, "normalization_std": std,
        "train_size": len(used_train), "validation_size": len(used_val),
        "split_sha256": hashlib.sha256(train_indices.tobytes() + val_indices.tobytes()).hexdigest(),
        "probe_indices": probe_indices.tolist(), "device": str(DEVICE),
        "torch": torch.__version__, "torchvision": torchvision.__version__,
        "python": platform.python_version(), "cost": cost,
        "probe_mode": "train, fixed unaugmented train batch, BN buffers restored"}
    (directory / "config.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

    def probe(epoch):
        snapshot = epoch in {0, TRAIN.epochs}
        sr, wr, rr, samples = state_diagnostics(model, initial, epoch, snapshot)
        states.extend(sr)
        weights.extend(wr)
        ratios.append(rr)
        if snapshot:
            np.savez_compressed(directory / f"histograms_epoch_{epoch:03d}.npz", **samples)

    def save_logs():
        for filename, records in [("history", history), ("states", states),
                                  ("weights", weights), ("ratios", ratios)]:
            pd.DataFrame(records).to_csv(directory / f"{filename}.csv", index=False)

    probe(0)
    save_logs()
    best_val, best_epoch = -1.0, None
    for epoch in range(1, TRAIN.epochs + 1):
        # Cùng seed/epoch ⇒ cùng shuffle + augmentation dù kiến trúc khác nhau.
        epoch_seed = seed * 100000 + epoch
        seed_all(epoch_seed)
        train_loader.generator.manual_seed(epoch_seed)
        start = time.perf_counter()
        lr = optimizer.param_groups[0]["lr"]
        train_metrics = train_epoch(model, train_loader, optimizer)
        validation = evaluate(model, val_loader)
        elapsed = time.perf_counter() - start
        history.append({"epoch": epoch, "lr": lr,
            "train_loss": train_metrics["loss"], "train_accuracy": train_metrics["accuracy"],
            "val_loss": validation["loss"], "val_accuracy": validation["accuracy"],
            "train_validation_seconds": elapsed})
        if validation["accuracy"] > best_val:
            best_val, best_epoch = validation["accuracy"], epoch
            torch.save({"model": {n: t.detach().cpu() for n, t in model.state_dict().items()},
                        "model_config": asdict(cfg), "epoch": epoch,
                        "val_accuracy": best_val}, directory / "best.pt")
        if epoch % TRAIN.probe_every == 0 or epoch == TRAIN.epochs:
            probe(epoch)
        scheduler.step()  # Gọi sau optimizer; LR trong CSV là LR thực dùng ở epoch.
        save_logs()
        print(f"{name} seed={seed} epoch={epoch:03d}/{TRAIN.epochs} lr={lr:.5f} "
              f"train={train_metrics['loss']:.4f}/{train_metrics['accuracy']:.3f} "
              f"val={validation['loss']:.4f}/{validation['accuracy']:.3f}", flush=True)
    return {"experiment": name, "seed": seed, "best_val_accuracy": best_val,
            "best_epoch": best_epoch, "directory": str(directory), **cost}


## 6. Ablation một yếu tố (tùy chọn)

Mặc định chỉ baseline. Đặt `RUN_ABLATIONS=True` và sửa `ABLATION_NAMES` ở cell cấu
hình để chọn thí nghiệm. Mỗi biến thể là bản sao baseline chỉ đổi đúng một field:
độ sâu 4/16/32, độ rộng 32/128, bottleneck16, sigmoid/tanh, Gaussian nhỏ/Xavier,
không norm/LayerNorm, không shortcut. Baseline đã đại diện depth8/width64/ReLU/He/BN/identity.
He luôn dùng gain ReLU kể cả ablation sigmoid/tanh để không đồng thời đổi activation
và khởi tạo; Xavier luôn gain=1. Giữ full batch, epochs, seeds, LR, data cho mọi run.

Shortcut cùng chiều dùng identity (không tham số). Projection chỉ dùng khi bottleneck
đổi chiều và được tính vào bảng parameters/MACs. So sánh norm `none` cần ghi chênh
lệch tham số affine đã nêu. Không đánh giá test để xếp hạng ablation.

Residual và BN có thể bảo vệ gradient ngay cả với sigmoid/weight nhỏ; nếu không thấy
gradient suy giảm, đó vẫn là kết quả cần báo cáo. Một tổ hợp plain+sigmoid sâu là
thí nghiệm tương tác khác, không được gắn nhãn là ablation một yếu tố của baseline này.
Notebook không triển khai RNN vì baseline report ở đây là CNN; nguyên nhân theo
khoảng cách thời gian cần notebook RNN và phép đo riêng.


In [ ]:
VARIANTS = {
    **{f"depth_{d}": replace(BASELINE, depth=d) for d in (4, 16, 32)},
    **{f"width_{w}": replace(BASELINE, width=w) for w in (32, 128)},
    "bottleneck_16": replace(BASELINE, bottleneck=16),
    "sigmoid": replace(BASELINE, activation="sigmoid"),
    "tanh": replace(BASELINE, activation="tanh"),
    "small_gaussian": replace(BASELINE, initialization="small_gaussian"),
    "xavier": replace(BASELINE, initialization="xavier"),
    "no_norm": replace(BASELINE, normalization="none"),
    "layernorm": replace(BASELINE, normalization="layernorm"),
    "no_shortcut": replace(BASELINE, residual=False),
}
for name, cfg in VARIANTS.items():
    changed = [key for key in asdict(BASELINE) if asdict(BASELINE)[key] != asdict(cfg)[key]]
    assert len(changed) == 1, (name, changed)
EXPERIMENTS = {"baseline": BASELINE}
if RUN_ABLATIONS:
    EXPERIMENTS.update({name: VARIANTS[name] for name in ABLATION_NAMES})
display(pd.DataFrame([{ "experiment": name, **asdict(cfg)} for name, cfg in EXPERIMENTS.items()]))


In [ ]:
# Cell huấn luyện chính. Report = 3 × 100 epoch cho mỗi cấu hình được chọn.
results = []
for name, cfg in EXPERIMENTS.items():
    for seed in TRAIN.seeds:
        results.append(run_experiment(name, cfg, seed))
        pd.DataFrame(results).to_csv(RUN_ROOT / "summary_per_seed.csv", index=False)
results_df = pd.DataFrame(results)
display(results_df)
summary = results_df.groupby("experiment").agg(
    val_accuracy_mean=("best_val_accuracy", "mean"),
    val_accuracy_std=("best_val_accuracy", "std"),
    parameters=("parameters", "first"),
    conv_linear_macs_per_image=("conv_linear_macs_per_image", "first"),
    seeds=("seed", "count"),
)
summary.to_csv(RUN_ROOT / "summary_validation.csv")
display(summary)
print("Quick chỉ có 1 seed ⇒ std=NaN; không thay NaN bằng 0.")


## 7. Biểu đồ để đưa vào report

Các đường theo epoch hiển thị mean ± std giữa seed. Profile gradient theo lớp
hiển thị từng seed ở khởi tạo và epoch cuối; trục log không vẽ giá trị 0/NaN.
Độ trôi weight và các proxy forward được hiển thị riêng vì chúng bổ sung thông tin
cho gradient trạng thái. Ảnh PNG được lưu tự động vào thư mục `figures`.


In [ ]:
def read_all(filename):
    frames = []
    for result in results:
        df = pd.read_csv(Path(result["directory"]) / filename)
        df["experiment"], df["seed"] = result["experiment"], result["seed"]
        frames.append(df)
    return pd.concat(frames, ignore_index=True)

history_df = read_all("history.csv")
states_df = read_all("states.csv")
weights_df = read_all("weights.csv")
ratios_df = read_all("ratios.csv")
FIG_ROOT = RUN_ROOT / "figures"
FIG_ROOT.mkdir(exist_ok=True)

def save_figure(fig, filename):
    fig.tight_layout()
    fig.savefig(FIG_ROOT / filename, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)

def mean_band(ax, df, metric, label, log=False):
    aggregate = df.groupby("epoch")[metric].agg(["mean", "std"])
    x, y = aggregate.index.to_numpy(), aggregate["mean"].to_numpy()
    s = aggregate["std"].fillna(0).to_numpy()  # Chỉ phục vụ vẽ; CSV std vẫn NaN.
    plotted = np.where(y > 0, y, np.nan) if log else y
    line, = ax.plot(x, plotted, label=label)
    if df["seed"].nunique() > 1:
        lo, hi = y - s, y + s
        if log:
            lo = np.where(lo > 0, lo, np.nan)
            hi = np.where(hi > 0, hi, np.nan)
        ax.fill_between(x, lo, hi, color=line.get_color(), alpha=0.15)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for name, frame in history_df.groupby("experiment", sort=False):
    mean_band(axes[0], frame, "train_loss", f"{name}: train")
    mean_band(axes[0], frame, "val_loss", f"{name}: val")
    mean_band(axes[1], frame, "val_accuracy", name)
axes[0].set(title="Learning curves", xlabel="Epoch", ylabel="Cross entropy")
axes[1].set(title="Validation accuracy", xlabel="Epoch", ylabel="Accuracy", ylim=(0, 1))
for ax in axes:
    ax.legend(fontsize=8)
save_figure(fig, "learning_curves.png")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for name, frame in ratios_df.groupby("experiment", sort=False):
    mean_band(axes[0], frame, "state_first_last_rms_ratio", name, log=True)
    mean_band(axes[1], frame, "weight_first_last_rms_ratio", name, log=True)
for ax, title in zip(axes, ["State gradient: stem / last block", "Weight gradient: stem / last Conv"]):
    ax.set(title=title, xlabel="Epoch", ylabel="RMS ratio", yscale="log")
    ax.axhline(1, color="gray", linestyle=":")
    ax.legend(fontsize=8)
save_figure(fig, "gradient_ratios.png")

for name, experiment in states_df.groupby("experiment", sort=False):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for epoch, ax in zip([0, TRAIN.epochs], axes):
        for seed, frame in experiment[experiment.epoch == epoch].groupby("seed"):
            frame = frame.sort_values("layer")
            y = frame.h_grad_rms.to_numpy()
            ax.plot(frame.layer, np.where(y > 0, y, np.nan), "o-", label=f"seed {seed}")
        ax.set(title=f"{name}: epoch {epoch}", xlabel="Layer (0=stem)",
               ylabel="RMS dL/dh", yscale="log")
        ax.legend()
    save_figure(fig, f"{name}_gradient_by_layer.png")

    last = experiment[experiment.epoch == TRAIN.epochs]
    branch = last[last.layer > 0]
    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    metrics = [("derivative_mean", "Mean activation derivative"),
               ("small_derivative_fraction", "Fraction derivative < 0.01"),
               ("h_batch_var", "Across-input activation variance"),
               ("h_pair_distance_rms", "Different-label pair distance RMS")]
    for ax, (metric, title) in zip(axes.flatten(), metrics):
        source = branch if metric.startswith(("derivative", "small")) else last
        for seed, frame in source.groupby("seed"):
            ax.plot(frame.layer, frame[metric], "o-", label=f"seed {seed}")
        ax.set(title=title, xlabel="Layer")
        ax.legend()
    save_figure(fig, f"{name}_activation_diagnostics.png")

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for seed, frame in weights_df[weights_df.experiment == name].groupby("seed"):
        for weight_name in ["stem.weight", f"blocks.{EXPERIMENTS[name].depth-1}.conv.weight"]:
            selected = frame[frame.name == weight_name]
            axes[0].plot(selected.epoch, selected.relative_drift, label=f"s{seed}: {weight_name}")
    for seed, frame in branch.groupby("seed"):
        axes[1].plot(frame.layer, frame.pre_norm_rms, "o-", label=f"s{seed}: before norm")
        axes[1].plot(frame.layer, frame.z_rms, "x--", label=f"s{seed}: after norm")
    axes[0].set(title=f"{name}: weight drift", xlabel="Epoch", ylabel="||W-W0|| / ||W0||")
    axes[1].set(title=f"{name}: preactivation scale, final epoch", xlabel="Block", ylabel="RMS")
    for ax in axes:
        ax.legend(fontsize=8)
    save_figure(fig, f"{name}_drift_and_signal_scale.png")


In [ ]:
# Histogram mẫu cố định (tối đa 4096 phần tử/tensor), seed đầu tiên.
for name, cfg in EXPERIMENTS.items():
    directory = RUN_ROOT / name / f"seed_{TRAIN.seeds[0]}"
    for epoch in (0, TRAIN.epochs):
        with np.load(directory / f"histograms_epoch_{epoch:03d}.npz") as snapshot:
            fig, axes = plt.subplots(1, 3, figsize=(15, 4))
            for layer in (1, cfg.depth):
                for ax, key, title in zip(axes, ("z", "a", "h_grad"),
                                          ("Activation input z", "Branch activation", "log10 |dL/dh|")):
                    values = snapshot[f"layer_{layer:02d}_{key}"]
                    if key == "h_grad":
                        values = np.log10(np.abs(values[values != 0]).astype(np.float64))
                    if len(values):
                        ax.hist(values, bins=50, alpha=0.5, density=True, label=f"block {layer}")
                    ax.set_title(title)
                    ax.set_ylabel("Density")
                    ax.legend(fontsize=8)
            fig.suptitle(f"{name}, seed {TRAIN.seeds[0]}, epoch {epoch}; gradient zeros omitted")
            save_figure(fig, f"{name}_histograms_epoch_{epoch:03d}.png")


## 8. Bảng chẩn đoán và cách viết nhận xét

Ngưỡng tỉ lệ `1e-4` dưới đây là **heuristic để ưu tiên kiểm tra**, không phải định
nghĩa chung của vanishing gradient. So sánh ở epoch 0, cuối và nhiều seed; nhìn cả
profile theo lớp và lượng gradient thực, tránh kết luận từ một batch hoặc loss chậm.
Phân biệt saturation sigmoid/tanh với ReLU bị chặn. Weight decay/momentum có thể
làm weight thay đổi dù data gradient nhỏ.

Gợi ý viết report: “Ở cấu hình A, gradient RMS stem/block cuối là … so với … của
baseline; derivative nhỏ chiếm …, variance activation là …; train loss ….
Kết quả phù hợp với cơ chế … trên các seed đã đo.” Nếu shortcut duy trì gradient
trạng thái dù đạo hàm nhánh nhỏ, ghi nhận hiệu quả của shortcut. Không khẳng định
baseline đã chứng minh tất cả nguyên nhân. Với norm/depth/width/bottleneck, kèm
số tham số, MACs và thời gian thực đo; không chỉ so accuracy.


In [ ]:
diagnostic_rows = []
for result in results:
    name, seed = result["experiment"], result["seed"]
    r = ratios_df[(ratios_df.experiment == name) & (ratios_df.seed == seed)].sort_values("epoch").iloc[-1]
    s = states_df[(states_df.experiment == name) & (states_df.seed == seed) & (states_df.epoch == r.epoch)]
    branch = s[s.layer > 0]
    w = weights_df[(weights_df.experiment == name) & (weights_df.seed == seed) & (weights_df.epoch == r.epoch)]
    observed_ratio = r.state_first_last_rms_ratio
    if not np.isfinite(observed_ratio):
        note = "Mẫu số gradient = 0 hoặc không xác định; kiểm tra từng lớp/underflow"
    elif observed_ratio < 1e-4:
        note = "Gradient trạng thái suy giảm mạnh về stem; kiểm tra profile và cơ chế"
    else:
        note = "Chưa thấy suy giảm mạnh theo heuristic trên probe này"
    diagnostic_rows.append({"experiment": name, "seed": seed,
        "state_ratio_final": observed_ratio,
        "weight_ratio_final": r.weight_first_last_rms_ratio,
        "mean_derivative_final": branch.derivative_mean.mean(),
        "small_derivative_fraction_final": branch.small_derivative_fraction.mean(),
        "stem_relative_drift_final": w[w.name == "stem.weight"].relative_drift.iloc[0],
        "max_state_gradient_zero_fraction": s.h_grad_zero_fraction.max(),
        "note": note})
diagnostic_df = pd.DataFrame(diagnostic_rows)
diagnostic_df.to_csv(RUN_ROOT / "diagnostics_per_seed.csv", index=False)
display(diagnostic_df)
numeric_metrics = ["state_ratio_final", "weight_ratio_final", "mean_derivative_final",
                   "small_derivative_fraction_final", "stem_relative_drift_final"]
diagnostic_summary = diagnostic_df.groupby("experiment")[numeric_metrics].agg(["mean", "std"])
diagnostic_summary.to_csv(RUN_ROOT / "diagnostics_mean_std.csv")
display(diagnostic_summary)


## 9. Đánh giá test sau khi hoàn tất chọn cấu hình

Mặc định đánh giá **baseline** ở report profile, dùng checkpoint được chọn bằng
validation của từng seed. Quick bỏ qua test. Nếu khảo sát nhiều cấu hình, lựa chọn
`SELECTED_EXPERIMENT` bằng validation và cố định lựa chọn trước khi chạy cell này;
không quay lại tuning dựa trên test. Đánh giá 10.000 ảnh test chính thức không augment.


In [ ]:
if EVALUATE_TEST:
    assert SELECTED_EXPERIMENT in EXPERIMENTS
    test_dataset = datasets.CIFAR10(str(DATA_ROOT), train=False, transform=eval_transform, download=True)
    assert len(test_dataset) == 10000
    test_loader = DataLoader(test_dataset, batch_size=TRAIN.batch_size, shuffle=False,
                             num_workers=TRAIN.num_workers)
    test_rows = []
    for seed in TRAIN.seeds:
        path = RUN_ROOT / SELECTED_EXPERIMENT / f"seed_{seed}" / "best.pt"
        checkpoint = torch.load(path, map_location="cpu", weights_only=True)
        model = DeepCNN(ModelConfig(**checkpoint["model_config"])).to(DEVICE)
        model.load_state_dict(checkpoint["model"])
        metrics = evaluate(model, test_loader)
        test_rows.append({"experiment": SELECTED_EXPERIMENT, "seed": seed,
            "selected_epoch": checkpoint["epoch"], "test_loss": metrics["loss"],
            "test_accuracy": metrics["accuracy"]})
        del model
    test_df = pd.DataFrame(test_rows)
    test_df.to_csv(RUN_ROOT / "test_per_seed.csv", index=False)
    display(test_df)
    test_summary = test_df.groupby("experiment")[["test_loss", "test_accuracy"]].agg(["mean", "std"])
    test_summary.to_csv(RUN_ROOT / "test_mean_std.csv")
    display(test_summary)
else:
    print("Bỏ qua test trong quick profile; đây chỉ là kiểm tra pipeline.")


## 10. File kết quả và tài liệu API

Trong thư mục run có:

- `split.npz`, `used_indices.npz`: split đầy đủ, tập con quick và batch probe.
- `summary_per_seed.csv`, `summary_validation.csv`: validation, parameters, MACs.
- `<experiment>/seed_<n>/config.json`: cấu hình thực, version, thống kê dữ liệu và split hash.
- `history.csv`: loss/accuracy, LR, thời gian train+validation (không gồm probe).
- `states.csv`: gradient trạng thái, signal/derivative/variance/khoảng cách theo lớp.
- `weights.csv`: norm/RMS data gradient, weight RMS, drift của Conv/Linear.
- `ratios.csv`: tỉ lệ stem/block cuối; riêng tỉ lệ stem/head để tránh nhầm mẫu số.
- `histograms_epoch_*.npz`, `best.pt`: mẫu histogram khởi tạo/cuối và checkpoint tốt nhất.
- `diagnostics_*.csv`, `test_*.csv` (nếu bật test), `figures/*.png`.

Tài liệu chính là report trong project. API đối chiếu:
[CIFAR10](https://docs.pytorch.org/vision/stable/generated/torchvision.datasets.CIFAR10.html),
[retain_grad](https://docs.pytorch.org/docs/2.9/generated/torch.Tensor.retain_grad.html),
[CosineAnnealingLR](https://docs.pytorch.org/docs/2.9/generated/torch.optim.lr_scheduler.CosineAnnealingLR.html).
Histogram/log là số đo của lần chạy thực; notebook không điền sẵn kết quả report
khi chưa huấn luyện đủ cấu hình quy định.


In [ ]:
print("Hoàn tất. Thư mục kết quả:", RUN_ROOT)
print("Biểu đồ:", FIG_ROOT)
